In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
import os
os.chdir("/content/gdrive/MyDrive/LLM/CBLLMMODEL")

In [ ]:
!pip install -q arch

In [ ]:
import pandas as pd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.tools import diff
from statsmodels.tsa.arima_model import ARMA, ARIMA, ARMAResults, ARIMAResults
from statsmodels.tsa.ar_model import AR, ARResults
from statsmodels.tsa.filters.hp_filter import hpfilter
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
from statsmodels.tsa.stattools import adfuller
import statsmodels.api as sm
from arch import arch_model
from itertools import combinations
warnings.filterwarnings("ignore")

In [ ]:
index = pd.date_range(start="2005-01-03", periods=7438, freq="D")
data = pd.read_excel("Dataset/ExchangeRate.xlsx")
data.fillna(method="ffill", inplace=True)
ExchangeRateDf = pd.DataFrame(index = index)
ExchangeRateDf["Value"] = data["USD"].values
ExchangeRateDf["Return"] = ExchangeRateDf.Value.pct_change(1)*100
ExchangeRateDf.head(3)

In [ ]:
ExchangeRateDf.Return.plot(figsize = (12,5), legend = True,
                title = "Return Value of Nominal Exchange Rate"
               )

In [ ]:
model_garch = arch_model(ExchangeRateDf.Return[1:], vol = "GARCH", mean = "Constant", p = 1, q = 2)
result_garch = model_garch.fit(6)
print(result_garch.summary())

In [ ]:
volatility = result_garch.conditional_volatility
volatility.plot(figsize = (12,3), title = "Model Volatility", legend = True, color = "red")

In [ ]:
ExchangeRateDf["Ex_Vol"] = volatility
ExchangeRateDf = ExchangeRateDf.dropna()
ExchangeRateDf.head()

In [ ]:
ExchangeRateDf1 = ExchangeRateDf.resample("M").mean()
ExchangeRateDf2 = ExchangeRateDf.resample("M").last()
ExchangeRateDf1.head()

In [ ]:
ExchangeRateDf2.head(3)

In [ ]:
ExchangeRateDf2[["Return", "Ex_Vol"]].plot(figsize = (12,3))

In [ ]:
Dataset = pd.read_excel("Dataset/MarkovDatasets.xlsx", index_col = "Date")
index = pd.date_range(start="2005-02-28", periods=242, freq="M")
Dataset.set_index(index, inplace=True)
Dataset[["Exc_Return1", "Exc_Volatility1"]] = ExchangeRateDf1[["Return", "Ex_Vol"]][1:-2]
Dataset[["Exc_Return2", "Exc_Volatility2"]] = ExchangeRateDf2[["Return", "Ex_Vol"]][1:-2]
Dataset = Dataset[10:]
Dataset.head(3)

In [ ]:
AveregaList = []
average = 4
Series = list(Dataset.CPI_M)

for i in range(len(Series) - average + 1 ):
    values = Series[i:i+average]
    avg = np.mean(values)
    AveregaList.append(avg)



In [ ]:
fig,ax = plt.subplots(figsize = (12,4))

ax.plot(AveregaList, color = "red")
ax.plot(Series[average-1:])

In [ ]:
bigger = 0
smaller = 0

for x,y in zip(AveregaList, Series[5:]):
    if x > y:
        smaller +=1
    else:
        bigger +=1

print(bigger)
print(smaller)

In [ ]:
ADF_Yearly = adfuller(Dataset["InterestRate"].diff().dropna())
print(f"Yearly ADF Statistic: {ADF_Yearly[0]}, p-value: {ADF_Yearly[1]}")

ADF_Yearly = adfuller(Dataset["Exc_Volatility1"])
print(f"Yearly ADF Statistic: {ADF_Yearly[0]}, p-value: {ADF_Yearly[1]}")

In [ ]:
exogs = [
       Dataset["PPI_M"].diff().dropna(),
       Dataset["NER"].diff().dropna(),
       Dataset["RER"].diff().dropna(),
       Dataset["IPE"].diff().dropna(),
       Dataset["InterestRate"].diff().dropna(),
       Dataset["Exc_Return1"][1:],
       Dataset["Exc_Volatility1"][1:],
       Dataset["Exc_Return2"][1:],
       Dataset["Exc_Volatility2"][1:]
           ]



# MARKOV REGRESSION

## Exog = 1, Regime = 2

In [ ]:
for exog in exogs:

    model = MarkovRegression(
                            endog = Dataset["CPI_M"].diff().dropna(),
                            k_regimes=2,
                            exog = exog,
                            switching_variance=True,
                            switching_trend=True,
                            switching_exog=True
                                            )

    result = model.fit()
    print(result.aic, exog.name)

## Exog = 1, Regime = 3

In [ ]:
for exog in exogs:

    model = MarkovRegression(
                            endog = Dataset["CPI_M"].diff().dropna(),
                            k_regimes=3,
                            exog = exog,
                            switching_variance=True,
                            # switching_trend=True,
                            switching_exog=True
                                            )
    result = model.fit()
    print(result.aic, exog.name)

## Exog = 2, Regime = 2

In [ ]:
for x in list(combinations(exogs, 2)):
    model = MarkovRegression(

                            endog = Dataset["CPI_M"].diff().dropna(),
                            k_regimes=2,
                            exog = pd.concat((x[0], x[1]), axis =1),
                            switching_variance=True,
                            switching_trend=True,
                            switching_exog=True
                                            )
    result = model.fit()
    print(result.aic, x[0].name, x[1].name)


## Exog = 2, Regime = 3

In [ ]:
for x in list(combinations(exogs, 2)):
    model = MarkovRegression(

                            endog = Dataset["CPI_M"].diff().dropna(),
                            k_regimes=3,
                            exog = pd.concat((x[0], x[1]), axis =1),
                            switching_variance=True,
                            switching_trend=True,
                            switching_exog=True
                                            )
    result = model.fit()
    print(result.aic, x[0].name, x[1].name)


# Exog = 3, Regime = 2

In [ ]:
for x in list(combinations(exogs, 3)):
    model = MarkovRegression(

                            endog = Dataset["CPI_M"].diff().dropna(),
                            k_regimes=2,
                            exog = pd.concat((x[0], x[1], x[2]), axis =1),
                            switching_variance=True,
                            switching_trend=True,
                            switching_exog=True
                                            )
    result = model.fit()
    print(result.aic, x[0].name, x[1].name, x[2].name)



## Exog=3, Regime = 3

In [ ]:
for x in list(combinations(exogs, 3)):
    model = MarkovRegression(

                            endog = Dataset["CPI_M"].diff().dropna(),
                            k_regimes=3,
                            exog = pd.concat((x[0], x[1], x[2]), axis =1),
                            switching_variance=True,
                            switching_trend=True,
                            switching_exog=True
                                            )
    result = model.fit()
    print(result.aic, x[0].name, x[1].name, x[2].name)


In [ ]:
model = MarkovRegression(

                            endog = Dataset["CPI_M"].diff().dropna(),
                            k_regimes=3,
                            switching_variance=True,
                            switching_trend=True,
                            switching_exog=True
                                            )
result = model.fit()
print(result.summary())

In [ ]:
import matplotlib.pyplot as plt

regime = 3

fig, axes = plt.subplots(regime, figsize=(10, 6), sharex=True)

for i in range(regime):
    probs = result.smoothed_marginal_probabilities[i]
    axes[i].plot(probs, color='blue')  # Çizgi rengi
    axes[i].fill_between(probs.index, probs, color='blue', alpha=0.1)  # Altını doldur
    axes[i].set_title(f"Regime {i} Probability")
    axes[i].set_ylabel("Probability")

axes[regime - 1].set_xlabel("Date")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

regime = 3
colors = ['darkblue', 'orange', 'red']  # Rejim renkleri

regimes = {0: "Moderate Inflation / Volatility",
           1: "Disinflation / Low Volatility",
           2: "High Volatility"}

fig, axes = plt.subplots(regime, figsize=(10, 6), sharex=True)

for i in range(regime):
    probs = result.smoothed_marginal_probabilities[i]
    axes[i].plot(probs, color="black", label=f'Regime {i}')
    axes[i].fill_between(probs.index, probs, color=colors[i], alpha=0.2)
    axes[i].set_title(regimes[i], fontweight = "bold")
    axes[i].set_ylabel("Probability", fontweight = "bold")
    axes[i].spines['top'].set_visible(False)
    axes[i].spines['right'].set_visible(False)

# axes[regime - 1].set_xlabel("Date")
plt.tight_layout()
plt.show()
fig.savefig("RegimeProbabilities.png", dpi=130, bbox_inches='tight')

In [ ]:
RegimeDf = result.smoothed_marginal_probabilities
RegimeDf["Regime"] = RegimeDf.idxmax(axis=1)
RegimeDf["Max_Prob"] = RegimeDf[[0, 1,2]].max(axis=1)
RegimeDf.to_excel("Dataset/RegimeDf.xlsx")
RegimeDf.head(5)

In [ ]:
RegimeDf["Regime"].value_counts()

In [ ]:
GovernorInterestDf = pd.read_excel("Dataset/GovernorInterestRate.xlsx", index_col = "Date")
GovernorInterestDf.head(3)

In [ ]:
RegimeDf["Regime2"] = [-1 if x == 1 else 1 if x == 0 else x for x in RegimeDf.Regime]
RegimeDf


In [ ]:
fig, ax = plt.subplots()

index = 1
for i in RegimeDf.Regime2:

    if x == 1:
        plt.plot([index, index], [0, i], "--bo")
    elif x == 2:
        plt.plot([index, index], [0, i], "--ko")
    else:
        plt.plot([index, index], [0, i], "--ro")
    index +=1


# RegimeDf.Regime2.plot()

# MARKOV AUTOREGRESSION

## Exog = 1, Regime = 2

In [ ]:
for exog in exogs:

    model_MSAR = sm.tsa.MarkovAutoregression(Dataset["CPI_M"].diff().dropna(),
                                            k_regimes=2,
                                            order = 1,
                                            trend='c',
                                            exog = exog,
                                            switching_variance=True,
                                            switching_exog=True)
    result_MSAR = model_MSAR.fit()
    print(result_MSAR.aic, exog.name)


## Exog = 1, Regime = 3

In [ ]:
for exog in exogs:

    model_MSAR = sm.tsa.MarkovAutoregression(Dataset["CPI_M"].diff().dropna(),
                                            k_regimes=3,
                                            order = 1,
                                            trend='c',
                                            exog = exog,
                                            switching_variance=True,
                                            switching_exog=True)
    result_MSAR = model_MSAR.fit()
    print(result_MSAR.aic, exog.name)

## Exog = 2, Regime = 2

In [ ]:
for x in list(combinations(exogs, 2)):

    model_MSAR = sm.tsa.MarkovAutoregression(Dataset["CPI_M"].diff().dropna(),
                                            k_regimes=2,
                                            order = 1,
                                            trend='c',
                                            exog = pd.concat((x[0], x[1]),
                                                              axis = 1),
                                            switching_variance=True,
                                            switching_exog=True)
    result_MSAR = model_MSAR.fit()
    print(result_MSAR.aic, x[0].name, x[1].name)

## Exog = 2, Regime = 3

In [ ]:
for x in list(combinations(exogs, 2)):

    model_MSAR = sm.tsa.MarkovAutoregression(Dataset["CPI_M"].diff().dropna(),
                                            k_regimes=3,
                                            order = 1,
                                            trend='c',
                                            exog = pd.concat((x[0], x[1]),
                                                              axis = 1),
                                            switching_variance=True,
                                            switching_exog=True)
    result_MSAR = model_MSAR.fit()
    print(result_MSAR.aic, x[0].name, x[1].name)

## Exog = 3, Regime = 2

In [ ]:
for x in list(combinations(exogs, 3)):

    model_MSAR = sm.tsa.MarkovAutoregression(Dataset["CPI_M"].diff().dropna(),
                                            k_regimes=2,
                                            order = 1,
                                            trend='c',
                                            exog = pd.concat((x[0], x[1], x[2]),
                                                              axis = 1),
                                            switching_variance=True,
                                            switching_exog=True)
    result_MSAR = model_MSAR.fit()
    print(result_MSAR.aic, x[0].name, x[1].name, x[2].name)

## Exog = 3, Regime = 3

In [ ]:
for x in list(combinations(exogs, 3)):

    model_MSAR = sm.tsa.MarkovAutoregression(Dataset["CPI_M"].diff().dropna(),
                                            k_regimes=3,
                                            order = 1,
                                            trend='c',
                                            exog = pd.concat((x[0], x[1], x[2]),
                                                              axis = 1),
                                            switching_variance=True,
                                            switching_exog=True)
    result_MSAR = model_MSAR.fit()
    print(result_MSAR.aic, x[0].name, x[1].name, x[2].name)

In [ ]:
model_MSAR = sm.tsa.MarkovAutoregression(Dataset["CPI_M"].diff().dropna(),
                                            k_regimes=3,
                                            order = 1,
                                            trend='c',
                                            exog = pd.concat((
                                                            Dataset["PPI_M"].diff().dropna(),
                                                              Dataset["NER"].diff().dropna(),
                                                            #    Dataset["Exc_Return"].diff().dropna(),


                                                               Dataset["Exc_Volatility1"][1:],
                                                            #    Dataset["Exc_Volatility"][1:],

                                                               ),
                                                              axis = 1),
                                            switching_variance=True,
                                         switching_exog = True)
result_MSAR = model_MSAR.fit()
print(result_MSAR.summary())

In [ ]:
import matplotlib.pyplot as plt

regime = 3

fig, axes = plt.subplots(regime, figsize=(10, 6), sharex=True)

for i in range(regime):
    probs = result_MSAR.smoothed_marginal_probabilities[i]
    axes[i].plot(probs, color='blue')  # Çizgi rengi
    axes[i].fill_between(probs.index, probs, color='blue', alpha=0.1)  # Altını doldur
    axes[i].set_title(f"Regime {i} Probability")
    axes[i].set_ylabel("Probability")

axes[regime - 1].set_xlabel("Date")
plt.tight_layout()
plt.show()


In [ ]:
RegimeDf = result_MSAR.smoothed_marginal_probabilities
RegimeDf.head()

In [ ]:
RegimeDf.tail()

In [ ]:
RegimeDf[0].plot(figsize = (12,4))
RegimeDf[1].plot(figsize = (12,4))
RegimeDf[2].plot(figsize = (12,4))

In [ ]:
RegimeDf["Regime"] = RegimeDf.idxmax(axis=1)
RegimeDf["Max_Prob"] = RegimeDf[[0, 1, 2]].max(axis=1)
RegimeDf.head(5)

In [ ]:
RegimeDf["Regime"].value_counts()

In [ ]:
import matplotlib.pyplot as plt

# Renk haritası
color_map = {0: "blue", 1: "orange", 2: "red"}

# Grafik çizimi
plt.figure(figsize=(18, 5))
bars = plt.bar(RegimeDf.index, RegimeDf["Max_Prob"],
               color=RegimeDf["Regime"].map(color_map),
               width=20)  # Genişliği ayarlayabilirsin

plt.title("Rejim Bazlı Maksimum Olasılık Grafiği")
plt.ylabel("Max_Prob")
plt.xlabel("Tarih")
plt.xticks(rotation=45)
plt.tight_layout()
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.show()

In [ ]:
fig, (ax1,ax2,ax3) = plt.subplots(3, figsize = (12,6))

for i in range(len(RegimeDf)):
    x1 = x2 = i
    y1 = 0
    y2 = RegimeDf.iloc[i]["Max_Prob"]

    regime = RegimeDf.iloc[i]["Regime"]

    if regime == 0:
        ax1.plot([x1,x2], [y1,y2], color= "blue")
    elif regime == 1:
        ax2.plot([x1,x2], [y1,y2],  color= "orange")
    else:
        ax3.plot([x1,x2], [y1,y2], color= "red" )

plt.tight_layout()
ax1.set_xlim(xmin = 0)
ax1.set_ylim(ymin = 0)

In [ ]:
RegimeDf.iloc[0]["Max_Prob"]

In [ ]:
x1 =  x2 = 2

In [ ]:
x1

In [ ]:
RegimeDf["Regime"].value_counts()

In [ ]:
RegimeDf.to_excel("InflationRegimes.xlsx")